# 01 · Ingesta y unificación (R1 · Hito 2)
**Proyecto semestral · Sala de crisis COVID-19 · Gestión de Datos IN1232C · 2026-2**

Qué hace este notebook (se ejecuta completo con *Ejecutar todo*):
1. **Descarga** reproducible de los 1.143 reportes diarios, las 5 series de tiempo y la tabla UID desde las URLs *raw* de GitHub → **capa cruda** (CSV tal cual, no se versiona).
2. **Unifica los cinco esquemas** en una sola tabla con nombres snake_case y fechas ISO, con columnas de linaje (archivo y fila de origen).
3. Escribe la **capa limpia** en Parquet (`data/clean/`).
4. **Compara** tamaño en disco y tiempo de lectura CSV vs Parquet.

**Dónde quedan los datos**
| Capa | Contenido | Ruta en Colab | ¿Va a GitHub? |
|---|---|---|---|
| Cruda | CSV originales | `/content/data/raw` (disco temporal de la sesión) | No: se regeneran con este notebook |
| Limpia | Parquet unificados | `<PROYECTO>/data/clean` | Sí |
| Analítica | DW (DuckDB) | `<PROYECTO>/data/dw` | Sí (notebook 03) |

En Colab `<PROYECTO>` es una carpeta de Google Drive, para que los notebooks siguientes (02, 03…) lean la capa limpia sin repetir la descarga. Fuera de Colab es la raíz del repositorio.

## 1 · Configuración

In [1]:
import sys, os, re, time
from pathlib import Path
import concurrent.futures as cf
import requests
import pandas as pd

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROYECTO = Path('/content/drive/MyDrive/proyecto-gestion-datos-2026')
    RAW = Path('/content/data/raw')                 # cruda: disco local de la sesión (rápido y desechable)
else:
    PROYECTO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    RAW = PROYECTO / 'data' / 'raw'

CLEAN = PROYECTO / 'data' / 'clean'
for p in [RAW / 'daily_reports', RAW / 'time_series', CLEAN]:
    p.mkdir(parents=True, exist_ok=True)
print('Proyecto:', PROYECTO, '| crudo:', RAW, '| limpio:', CLEAN)

Proyecto: /home/claude/facuna-m/proyecto-gestion-datos-2026 | crudo: /home/claude/facuna-m/proyecto-gestion-datos-2026/data/raw | limpio: /home/claude/facuna-m/proyecto-gestion-datos-2026/data/clean


## 2 · Descarga reproducible (capa cruda)

In [2]:
URL = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/'
FECHAS = pd.date_range('2020-01-22', '2023-03-09', freq='D')   # repositorio archivado: no cambia
SERIES = ['time_series_covid19_confirmed_global.csv', 'time_series_covid19_deaths_global.csv',
          'time_series_covid19_recovered_global.csv', 'time_series_covid19_confirmed_US.csv',
          'time_series_covid19_deaths_US.csv']

def descargar(url: str, destino: Path, reintentos: int = 3) -> str:
    """Descarga si el archivo no existe. Devuelve 'ok', 'ya estaba' o el error."""
    if destino.exists() and destino.stat().st_size > 0:
        return 'ya estaba'
    for intento in range(reintentos):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            destino.write_bytes(r.content)
            return 'ok'
        except Exception as e:
            err = str(e); time.sleep(2 * (intento + 1))
    return 'ERROR: ' + err

tareas = [(URL + 'csse_covid_19_daily_reports/' + f'{d:%m-%d-%Y}.csv', RAW / 'daily_reports' / f'{d:%m-%d-%Y}.csv') for d in FECHAS]
tareas += [(URL + 'csse_covid_19_time_series/' + s, RAW / 'time_series' / s) for s in SERIES]
tareas += [(URL + 'UID_ISO_FIPS_LookUp_Table.csv', RAW / 'UID_ISO_FIPS_LookUp_Table.csv')]

t0 = time.time()
with cf.ThreadPoolExecutor(max_workers=16) as ex:          # 16 descargas en paralelo
    estados = list(ex.map(lambda t: descargar(*t), tareas))
print(f'{len(tareas)} archivos en {time.time() - t0:.0f} s ->', pd.Series(estados).value_counts().to_dict())
assert not any(e.startswith('ERROR') for e in estados), 'Hubo descargas fallidas: vuelve a ejecutar la celda'
assert len(list((RAW / 'daily_reports').glob('*.csv'))) == 1143

1149 archivos en 0 s -> {'ya estaba': 1149}


## 3 · Detección de esquemas
Agrupamos los archivos por su lista exacta de columnas: deben aparecer las **cinco etapas** del enunciado.

In [3]:
archivos = sorted((RAW / 'daily_reports').glob('*.csv'), key=lambda p: pd.to_datetime(p.stem, format='%m-%d-%Y'))
esquemas = {}
for p in archivos:
    esquemas.setdefault(tuple(pd.read_csv(p, nrows=0).columns), []).append(p.stem)

ETAPA_DE_ARCHIVO = {}
tabla = []
for i, (cols, fs) in enumerate(esquemas.items(), start=1):
    for f in fs: ETAPA_DE_ARCHIVO[f] = i
    tabla.append({'etapa': i, 'desde': fs[0], 'hasta': fs[-1], 'archivos': len(fs), 'n_columnas': len(cols), 'columnas': ', '.join(cols)})
pd.DataFrame(tabla).set_index('etapa')

,desde,hasta,archivos,n_columnas,columnas
etapa,,,,,
1,01-22-2020,02-29-2020,39,6,"Province/State, Country/Region, Last Update, C..."
2,03-01-2020,03-21-2020,21,8,"Province/State, Country/Region, Last Update, C..."
3,03-22-2020,05-28-2020,68,12,"FIPS, Admin2, Province_State, Country_Region, ..."
4,05-29-2020,11-08-2020,164,14,"FIPS, Admin2, Province_State, Country_Region, ..."
5,11-09-2020,03-09-2023,851,14,"FIPS, Admin2, Province_State, Country_Region, ..."


## 4 · Unificación de los cinco esquemas
Reglas de unificación (decisiones documentadas):

| Problema | Decisión |
|---|---|
| Estilos de nombre distintos (`Country/Region`, `Country_Region`, `Last Update`, `Long_`, `Case-Fatality_Ratio`) | Función `a_snake_case` |
| Etapa 2 usa `Latitude`/`Longitude`; etapa 3+ usa `Lat`/`Long_` | `latitude → lat`, `longitude → long` |
| Etapa 4 `Incidence_Rate`, etapa 5 `Incident_Rate` | Se unifican en `incident_rate` |
| `Last Update` en ≥4 formatos de fecha | `pd.to_datetime(format='mixed')`, se guarda como `last_update` (fecha-hora) |
| ¿Qué fecha representa la fila? | `fecha_reporte` = fecha del **nombre del archivo** (el corte diario). Es la fecha del grano del DW |
| Conteos con vacíos | Se guardan como enteros *nullable* (`Int64`): **no se imputan** ceros en esta capa |
| Espacios sobrantes (`' Azerbaijan'`) | `strip()` en textos |
| Linaje | `archivo_origen`, `fila_origen` (n.º de fila de datos en el CSV, desde 1) y `etapa_esquema` |

Los nombres de país **no** se homologan aquí (eso se hace con la tabla UID en la capa analítica): la capa limpia conserva el valor original para no perder trazabilidad.

In [4]:
def a_snake_case(nombre: str) -> str:
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', nombre.strip())
    s = re.sub(r'[^0-9a-zA-Z]+', '_', s)
    return s.strip('_').lower()

RENOMBRAR = {'latitude': 'lat', 'longitude': 'long', 'incidence_rate': 'incident_rate'}
COLUMNAS = ['fecha_reporte', 'fips', 'admin2', 'province_state', 'country_region', 'combined_key',
            'lat', 'long', 'confirmed', 'deaths', 'recovered', 'active', 'incident_rate',
            'case_fatality_ratio', 'last_update', 'etapa_esquema', 'archivo_origen', 'fila_origen']
TEXTO = ['admin2', 'province_state', 'country_region', 'combined_key']
CONTEOS = ['confirmed', 'deaths', 'recovered', 'active']

def leer_unificado(ruta: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, na_values=[''])   # todo como texto: casteo explícito
    df.columns = [a_snake_case(c) for c in df.columns]
    df = df.rename(columns=RENOMBRAR).reindex(columns=COLUMNAS)   # columnas que no existían ese día quedan como NaN
    df['fecha_reporte'] = pd.to_datetime(ruta.stem, format='%m-%d-%Y')
    df['etapa_esquema'] = ETAPA_DE_ARCHIVO[ruta.stem]
    df['archivo_origen'] = ruta.name
    df['fila_origen'] = range(1, len(df) + 1)
    # Casteo de tipos archivo por archivo (así la memoria no explota al concatenar ~4 millones de filas)
    for c in TEXTO:
        df[c] = df[c].astype('string').str.strip()   # 'string' también funciona si la columna no existía (toda NaN)
    for c in CONTEOS + ['fips']:
        df[c] = pd.to_numeric(df[c], errors='coerce').round().astype('Int64')
    for c in ['lat', 'long', 'incident_rate', 'case_fatality_ratio']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['last_update'] = pd.to_datetime(df['last_update'], format='mixed', errors='coerce')
    df['etapa_esquema'] = df['etapa_esquema'].astype('int8')
    df['fila_origen'] = df['fila_origen'].astype('int32')
    return df

t0 = time.time()
covid = pd.concat([leer_unificado(p) for p in archivos], ignore_index=True)
print(f'Lectura, casteo y concatenación: {time.time() - t0:.0f} s · {covid.shape[0]:,} filas')

Lectura, casteo y concatenación: 32 s · 4,287,473 filas


In [5]:
for c in TEXTO + ['archivo_origen']:
    covid[c] = covid[c].astype('category')     # pocas categorías repetidas millones de veces -> mucha menos memoria

print('Fechas last_update no interpretables:', covid['last_update'].isna().sum())
covid.info(memory_usage='deep')

Fechas last_update no interpretables: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4287473 entries, 0 to 4287472
Data columns (total 18 columns):
 #   Column               Dtype         
---  ------               -----         
 0   fecha_reporte        datetime64[ns]
 1   fips                 Int64         
 2   admin2               category      
 3   province_state       category      
 4   country_region       category      
 5   combined_key         category      
 6   lat                  float64       
 7   long                 float64       
 8   confirmed            Int64         
 9   deaths               Int64         
 10  recovered            Int64         
 11  active               Int64         
 12  incident_rate        float64       
 13  case_fatality_ratio  float64       
 14  last_update          datetime64[ns]
 15  etapa_esquema        int8          
 16  archivo_origen       category      
 17  fila_origen          int32         
dtypes: Int64(5), category(

### Verificaciones de la unificación

In [6]:
filas_csv = sum(sum(1 for _ in open(p, encoding='utf-8-sig')) - 1 for p in archivos)   # filas de datos en los CSV
print(f'Filas en los 1.143 CSV: {filas_csv:,} | filas en la tabla unificada: {len(covid):,}')
assert filas_csv == len(covid), 'Se perdieron o duplicaron filas en la unificación'
assert covid['fecha_reporte'].nunique() == 1143
assert list(covid.columns) == COLUMNAS

resumen_etapas = covid.groupby('etapa_esquema').agg(desde=('fecha_reporte', 'min'), hasta=('fecha_reporte', 'max'),
                                                    archivos=('archivo_origen', 'nunique'), filas=('fila_origen', 'size'))
resumen_etapas

Filas en los 1.143 CSV: 4,287,473 | filas en la tabla unificada: 4,287,473


,desde,hasta,archivos,filas
etapa_esquema,,,,
1,2020-01-22,2020-02-29,39,4242
2,2020-03-01,2020-03-21,21,5555
3,2020-03-22,2020-05-28,68,217486
4,2020-05-29,2020-11-08,164,643463
5,2020-11-09,2023-03-09,851,3416727


In [7]:
# Disponibilidad de cada columna por etapa (% no nulo): confirma que la unificación no dejó columnas 'huérfanas'
(covid.drop(columns=['fecha_reporte', 'etapa_esquema', 'archivo_origen', 'fila_origen'])
      .notna().groupby(covid['etapa_esquema']).mean().mul(100).round(1).T)

etapa_esquema,1,2,3,4,5
fips,0.0,0.0,89.6,81.7,81.4
admin2,0.0,0.0,89.7,81.8,81.5
province_state,73.0,53.9,94.2,95.5,95.5
country_region,100.0,100.0,100.0,100.0,100.0
combined_key,0.0,0.0,100.0,100.0,100.0
lat,0.0,98.9,98.2,97.9,97.7
long,0.0,98.9,98.2,97.9,97.7
confirmed,99.3,100.0,100.0,100.0,100.0
deaths,89.8,100.0,100.0,100.0,100.0
recovered,90.9,100.0,100.0,100.0,17.2


In [8]:
# Ejemplo de linaje: de dónde sale el total de Chile del 17-jun-2020 (pico por rezagos)
covid.loc[(covid['country_region'] == 'Chile') & (covid['fecha_reporte'] == '2020-06-17'),
          ['fecha_reporte', 'province_state', 'confirmed', 'deaths', 'archivo_origen', 'fila_origen']]

,fecha_reporte,province_state,confirmed,deaths,archivo_origen,fila_origen
300777,2020-06-17,Antofagasta,5241,81,06-17-2020.csv,3093
300782,2020-06-17,Araucania,2815,36,06-17-2020.csv,3098
300784,2020-06-17,Arica y Parinacota,1218,11,06-17-2020.csv,3100
300791,2020-06-17,Atacama,561,0,06-17-2020.csv,3107
300795,2020-06-17,Aysen,26,0,06-17-2020.csv,3111
300811,2020-06-17,Biobio,4015,25,06-17-2020.csv,3127
300860,2020-06-17,Coquimbo,1980,12,06-17-2020.csv,3176
301000,2020-06-17,Los Lagos,1249,14,06-17-2020.csv,3316
301001,2020-06-17,Los Rios,555,10,06-17-2020.csv,3317
301009,2020-06-17,Magallanes,1284,15,06-17-2020.csv,3325


## 5 · Series de tiempo y tabla UID
Las series vienen en **formato ancho** (una columna por día). Se pasan a **formato largo** (territorio × fecha), que es el grano del DW y permite comparar con la suma de los reportes diarios (R2).

In [9]:
def serie_a_largo(nombre: str) -> pd.DataFrame:
    df = pd.read_csv(RAW / 'time_series' / nombre)
    df = df.drop(columns=[c for c in ['iso2', 'iso3', 'code3', 'Combined_Key', 'Population'] if c in df.columns])  # ya están en la tabla UID
    df.columns = [c if re.match(r'\d+/\d+/\d+', c) else a_snake_case(c) for c in df.columns]
    fechas = [c for c in df.columns if re.match(r'\d+/\d+/\d+', c)]
    ids = [c for c in df.columns if c not in fechas]
    largo = df.melt(id_vars=ids, value_vars=fechas, var_name='fecha', value_name='valor')
    largo['fecha'] = pd.to_datetime(largo['fecha'], format='%m/%d/%y')
    largo['valor'] = largo['valor'].astype('Int64')
    return largo, len(fechas)

import gc
for s in SERIES:
    clave = s.replace('time_series_covid19_', '').replace('.csv', '')
    largo, n_fechas = serie_a_largo(s)
    largo.to_parquet(CLEAN / f'ts_{clave}.parquet', index=False)
    print(f'{clave:18s} {n_fechas} columnas de fecha -> {len(largo):>10,} filas en formato largo')
    del largo; gc.collect()                      # liberar memoria: la serie ya quedó en Parquet
pd.read_parquet(CLEAN / 'ts_confirmed_global.parquet').query("country_region == 'Chile'").tail()

confirmed_global   1143 columnas de fecha ->    330,327 filas en formato largo
deaths_global      1143 columnas de fecha ->    330,327 filas en formato largo


recovered_global   1143 columnas de fecha ->    313,182 filas en formato largo


confirmed_US       1143 columnas de fecha ->  3,819,906 filas en formato largo


deaths_US          1143 columnas de fecha ->  3,819,906 filas en formato largo


,province_state,country_region,lat,long,fecha,valor
328940,None,Chile,-35.6751,-71.543,2023-03-05,5180329
329229,None,Chile,-35.6751,-71.543,2023-03-06,5182360
329518,None,Chile,-35.6751,-71.543,2023-03-07,5184582
329807,None,Chile,-35.6751,-71.543,2023-03-08,5188380
330096,None,Chile,-35.6751,-71.543,2023-03-09,5192286


In [10]:
uid = pd.read_csv(RAW / 'UID_ISO_FIPS_LookUp_Table.csv', dtype={'FIPS': 'Int64', 'Population': 'Int64', 'code3': 'Int64'})
uid.columns = [a_snake_case(c) for c in uid.columns]
uid.to_parquet(CLEAN / 'uid_lookup.parquet', index=False)
print(uid.shape, '| territorios con población:', uid['population'].notna().sum())
uid[uid['country_region'] == 'Chile']

(4321, 12) | territorios con población: 4170


,uid,iso2,iso3,code3,fips,admin2,province_state,country_region,lat,long,combined_key,population
74,152,CL,CHL,152,<NA>,NaN,NaN,Chile,-35.6751,-71.5430,Chile,19116209
75,15201,CL,CHL,152,<NA>,NaN,Antofagasta,Chile,-23.6509,-70.3975,"Antofagasta, Chile",607534
76,15202,CL,CHL,152,<NA>,NaN,Araucania,Chile,-38.9489,-72.3311,"Araucania, Chile",957224
77,15203,CL,CHL,152,<NA>,NaN,Arica y Parinacota,Chile,-18.5940,-69.4785,"Arica y Parinacota, Chile",226068
78,15204,CL,CHL,152,<NA>,NaN,Atacama,Chile,-27.5661,-70.0503,"Atacama, Chile",288944
79,15205,CL,CHL,152,<NA>,NaN,Aysen,Chile,-45.9864,-73.7669,"Aysen, Chile",103158
80,15206,CL,CHL,152,<NA>,NaN,Biobio,Chile,-37.4464,-72.1416,"Biobio, Chile",1556805
81,15207,CL,CHL,152,<NA>,NaN,Coquimbo,Chile,-29.9590,-71.3389,"Coquimbo, Chile",757586
82,15208,CL,CHL,152,<NA>,NaN,Los Lagos,Chile,-41.9198,-72.1416,"Los Lagos, Chile",828708
83,15209,CL,CHL,152,<NA>,NaN,Los Rios,Chile,-40.2310,-72.3311,"Los Rios, Chile",384837


## 6 · Capa limpia en Parquet

In [11]:
t0 = time.time()
covid.to_parquet(CLEAN / 'daily_reports.parquet', index=False, compression='zstd')
print(f'Escritura Parquet: {time.time() - t0:.1f} s')
for p in sorted(CLEAN.glob('*.parquet')):
    print(f'  {p.name:35s} {p.stat().st_size / 1e6:7.1f} MB')

Escritura Parquet: 2.0 s
  daily_reports.parquet                  70.6 MB
  ts_confirmed_US.parquet                23.4 MB
  ts_confirmed_global.parquet             1.4 MB
  ts_deaths_US.parquet                   19.3 MB
  ts_deaths_global.parquet                0.8 MB
  ts_recovered_global.parquet             0.7 MB
  uid_lookup.parquet                      0.2 MB


## 7 · CSV vs Parquet: tamaño y tiempo de lectura
Se comparan tres formas de guardar **los mismos datos**:
* **CSV crudos**: los 1.143 archivos originales.
* **CSV unificado**: la tabla limpia guardada como un solo CSV.
* **Parquet** (columnar, comprimido con zstd, conserva los tipos).

In [12]:
csv_unico = Path('/tmp/daily_reports_unificado.csv')
covid.to_csv(csv_unico, index=False)
del covid; gc.collect()          # ya está en Parquet; liberamos RAM para medir lecturas en igualdad de condiciones

def medir(func, repeticiones=2):
    """Mejor tiempo (s) de varias lecturas; el resultado se descarta para no acumular memoria."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter(); func(); tiempos.append(time.perf_counter() - t0); gc.collect()
    return min(tiempos)

t_crudos = medir(lambda: len(pd.concat([pd.read_csv(p) for p in archivos])), repeticiones=1)
t_csv = medir(lambda: len(pd.read_csv(csv_unico, low_memory=False)))
t_parq = medir(lambda: len(pd.read_parquet(CLEAN / 'daily_reports.parquet')))
t_parq_cols = medir(lambda: len(pd.read_parquet(CLEAN / 'daily_reports.parquet', columns=['fecha_reporte', 'country_region', 'confirmed'])))

comparacion = pd.DataFrame({
    'formato': ['CSV crudos (1.143 archivos)', 'CSV unificado (1 archivo)', 'Parquet zstd', 'Parquet · solo 3 columnas'],
    'tamaño_MB': [sum(p.stat().st_size for p in archivos) / 1e6, csv_unico.stat().st_size / 1e6,
                  (CLEAN / 'daily_reports.parquet').stat().st_size / 1e6, None],
    'lectura_s': [t_crudos, t_csv, t_parq, t_parq_cols],
}).set_index('formato')
comparacion['tamaño_vs_parquet'] = (comparacion['tamaño_MB'] / comparacion.loc['Parquet zstd', 'tamaño_MB']).round(1)
comparacion['lectura_vs_parquet'] = (comparacion['lectura_s'] / comparacion.loc['Parquet zstd', 'lectura_s']).round(1)
csv_unico.unlink()
comparacion.round(2)

,tamaño_MB,lectura_s,tamaño_vs_parquet,lectura_vs_parquet
formato,,,,
CSV crudos (1.143 archivos),587.06,8.66,8.3,21.9
CSV unificado (1 archivo),726.22,8.76,10.3,22.1
Parquet zstd,70.59,0.40,1.0,1.0
Parquet · solo 3 columnas,NaN,0.11,NaN,0.3


**Conclusión (ejecución de referencia; los tiempos exactos varían según la máquina):**
* **Tamaño:** los 1.143 CSV ocupan ~587 MB; la misma información en Parquet ocupa ~71 MB (**~8 veces menos**). El CSV unificado es aún más grande (~726 MB) porque repite en texto columnas que antes no existían.
* **Lectura:** leer todo el Parquet toma ~0,5 s frente a ~9 s del CSV (**~17 veces más rápido**), y además llega con los tipos correctos (fechas, enteros, categorías) sin volver a convertir.
* **Lectura selectiva:** por ser columnar, Parquet puede leer solo las columnas necesarias (~0,1 s para 3 columnas). Un CSV siempre se lee completo.
* Por eso la capa limpia es Parquet y el DW (DuckDB) puede consultarla directamente con SQL.

## 8 · Resumen
* **Capa cruda**: 1.143 reportes + 5 series + tabla UID descargados con una celda (reintentos y descarga paralela; idempotente).
* **Capa limpia**: `daily_reports.parquet` con los 5 esquemas unificados, tipos explícitos y linaje por fila; series en formato largo y tabla UID en Parquet.
* Próximo paso: `02_calidad` (reglas de las 6 dimensiones) y `03_dw` (modelo en estrella en DuckDB).